# Session 5 — Options Pricing: Black-Scholes, Monte Carlo & Real Options
## Risk Modelling Course

---

### Learning Objectives
By the end of this session you will be able to:
1. Implement the **Black-Scholes formula** analytically in Python and understand each component
2. Compute and visualise the **Greeks** (Delta, Gamma, Vega, Theta, Rho)
3. Price European options by **Monte Carlo simulation** (GBM path simulation)
4. Price **path-dependent options** (Asian options) that have no closed-form solution
5. Demonstrate that MC prices **converge** to the BS price as simulation paths increase
6. Apply **implied volatility** concepts and understand the volatility smile
7. Introduce **Real Options** — frame corporate investments as financial options

---

### Core idea — Geometric Brownian Motion (GBM)
Black-Scholes assumes the underlying asset follows:

$$dS = \mu S \, dt + \sigma S \, dW_t$$

Where $dW_t = \epsilon \sqrt{dt}$ is a Wiener process increment ($\epsilon \sim N(0,1)$).

In discrete time (for simulation) this becomes:

$$S_{t+\Delta t} = S_t \exp\!\left[\left(r - \tfrac{1}{2}\sigma^2\right)\Delta t + \sigma \sqrt{\Delta t}\, \epsilon\right]$$

The **risk-neutral** version replaces $\mu$ with $r$ (the risk-free rate) — a foundational result.


## 0. Imports & Configuration

In [ ]:

import warnings
warnings.filterwarnings("ignore")

# ── Numerics ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq   # for implied vol solver

# ── Market data ───────────────────────────────────────────────────────────────
import yfinance as yf

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10", font_scale=1.15)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Reproducibility ───────────────────────────────────────────────────────────
np.random.seed(42)

print("All libraries loaded.")


## 1. The Black-Scholes Formula (Analytical)

The Black-Scholes formula prices a **European** option (can only be exercised at expiry):

$$C = S_0 \cdot N(d_1) - K e^{-rT} \cdot N(d_2)$$
$$P = K e^{-rT} \cdot N(-d_2) - S_0 \cdot N(-d_1)$$

Where:
$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

- $S_0$ = current stock price
- $K$ = strike price
- $T$ = time to maturity (in years)
- $r$ = continuously compounded risk-free rate
- $\sigma$ = annual volatility of the underlying
- $N(\cdot)$ = standard normal CDF


In [ ]:

def black_scholes(S: float, K: float, T: float, r: float,
                  sigma: float, option: str = "call") -> dict:
    """
    Compute Black-Scholes price and Greeks for a European option.

    Parameters
    ----------
    S      : Current stock price
    K      : Strike price
    T      : Time to maturity in years
    r      : Continuously compounded risk-free rate (annual)
    sigma  : Annual volatility (e.g. 0.20 for 20%)
    option : 'call' or 'put'

    Returns
    -------
    dict with keys: price, d1, d2, delta, gamma, vega, theta, rho
    """
    # ── Protect against T=0 ───────────────────────────────────────────────────
    if T <= 0:
        intrinsic = max(S - K, 0) if option == "call" else max(K - S, 0)
        return {"price": intrinsic}

    # ── d1 and d2 ─────────────────────────────────────────────────────────────
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # ── Price ──────────────────────────────────────────────────────────────────
    if option == "call":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    # ── Greeks ────────────────────────────────────────────────────────────────
    # Delta: sensitivity of price to ±$1 move in S
    delta = norm.cdf(d1) if option == "call" else norm.cdf(d1) - 1

    # Gamma: rate of change of Delta w.r.t. S (same for calls and puts)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))

    # Vega: sensitivity to ±1% change in sigma (reported per 1% move)
    vega  = S * norm.pdf(d1) * np.sqrt(T) / 100

    # Theta: time decay per calendar day (reported as daily loss)
    if option == "call":
        theta = (
            -S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
            - r * K * np.exp(-r * T) * norm.cdf(d2)
        ) / 365
    else:
        theta = (
            -S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
            + r * K * np.exp(-r * T) * norm.cdf(-d2)
        ) / 365

    # Rho: sensitivity to ±1% change in risk-free rate
    if option == "call":
        rho = K * T * np.exp(-r * T) * norm.cdf(d2)  / 100
    else:
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    return {
        "price": price, "d1": d1, "d2": d2,
        "delta": delta, "gamma": gamma,
        "vega":  vega,  "theta": theta, "rho": rho
    }


# ── Example pricing ───────────────────────────────────────────────────────────
# AAPL-like parameters: stock at 180, ATM call, 3 months to expiry, 20% vol
params = dict(S=180, K=180, T=0.25, r=0.05, sigma=0.20)

call_result = black_scholes(**params, option="call")
put_result  = black_scholes(**params, option="put")

print(f"{'':=<55}")
print(f"  Black-Scholes Pricing Example")
print(f"  S={params['S']}, K={params['K']}, T={params['T']}yr, r={params['r']:.0%}, σ={params['sigma']:.0%}")
print(f"{'':=<55}")
for key in ["price", "delta", "gamma", "vega", "theta", "rho"]:
    print(f"  CALL {key:>6}: {call_result[key]:>9.4f}    PUT {key:>6}: {put_result[key]:>9.4f}")
print(f"{'':=<55}")
print()
print("Put-Call Parity check (C - P should equal S - K·e^{-rT}):")
lhs = call_result["price"] - put_result["price"]
rhs = params["S"] - params["K"] * np.exp(-params["r"] * params["T"])
print(f"  C - P = {lhs:.4f}   |   S - Ke^(-rT) = {rhs:.4f}   ✓" if abs(lhs-rhs) < 1e-8
      else f"  MISMATCH: {lhs:.4f} vs {rhs:.4f}")


## 2. Visualising the Greeks

Greeks tell traders *how* the option price changes with each input.
Understanding them is essential for **hedging** (Delta hedging) and **risk management**.

| Greek | What it measures | Intuition |
|-------|-----------------|-----------|
| Delta | ∂Price/∂S | Like a hedge ratio — a delta of 0.5 means price moves 50p per £1 in stock |
| Gamma | ∂²Price/∂S² | Rate of change of delta; largest near ATM options |
| Vega  | ∂Price/∂σ | How much the option gains/loses per 1% vol rise |
| Theta | ∂Price/∂t | Time decay — options lose value as expiry approaches |


In [ ]:

# ── Parameter grid: vary spot price from 50% to 150% of strike ────────────────
K, T, r, sigma = 100, 0.5, 0.05, 0.25
spot_range = np.linspace(50, 150, 300)

# Build a DataFrame for call and put Greeks across spot prices using chaining
def greeks_df(option_type):
    """Compute all Greeks across a spot price range."""
    records = [black_scholes(S=s, K=K, T=T, r=r, sigma=sigma, option=option_type)
               for s in spot_range]
    return (
        pd.DataFrame(records)
        .assign(Spot=spot_range, Option=option_type)
    )

greeks_all = pd.concat([greeks_df("call"), greeks_df("put")], ignore_index=True)

# ── 4-panel plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
greek_specs = [
    ("price", "Option Price (£)",         "Price vs Spot"),
    ("delta", "Delta",                     "Delta vs Spot"),
    ("gamma", "Gamma",                     "Gamma vs Spot (identical for C & P)"),
    ("vega",  "Vega (£ per 1% σ move)",   "Vega vs Spot"),
]

for ax, (greek, ylabel, title) in zip(axes.flatten(), greek_specs):
    for opt_type, grp in greeks_all.groupby("Option"):
        ax.plot(grp["Spot"], grp[greek], lw=2,
                label=opt_type.capitalize(),
                ls="-" if opt_type == "call" else "--")

    ax.axvline(K, color="grey", lw=1, ls=":", label=f"Strike K={K}")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Spot Price (S)")
    ax.set_ylabel(ylabel)
    ax.legend()

fig.suptitle(f"Black-Scholes Greeks  |  K={K}, T={T}yr, r={r:.0%}, σ={sigma:.0%}",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 3. Simulating Stock Price Paths with Geometric Brownian Motion (GBM)

Before pricing options by simulation, let's visualise what GBM looks like.

The key discretisation (Euler-Maruyama scheme):
$$S_{t+1} = S_t \exp\!\left[(r - \tfrac{1}{2}\sigma^2)\,\Delta t + \sigma\sqrt{\Delta t}\,\epsilon_t\right]$$

We use **vectorised NumPy** rather than Python loops — this is critical for speed when
running 100,000 paths.


In [ ]:

def simulate_gbm(S0: float, r: float, sigma: float,
                 T: float, n_steps: int, n_paths: int) -> np.ndarray:
    """
    Simulate GBM paths using vectorised NumPy operations.

    Returns
    -------
    np.ndarray of shape (n_steps + 1, n_paths)
    Each column is one simulated price path.
    """
    dt = T / n_steps

    # ── Draw all random increments at once (much faster than looping) ─────────
    # Shape: (n_steps, n_paths)
    Z = np.random.standard_normal((n_steps, n_paths))

    # ── Compute log-return increments ─────────────────────────────────────────
    increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z

    # ── Cumulative sum and exponentiation to recover price levels ──────────────
    # np.vstack prepends a row of zeros for S0
    log_paths = np.vstack([np.zeros(n_paths), np.cumsum(increments, axis=0)])
    paths = S0 * np.exp(log_paths)   # shape: (n_steps + 1, n_paths)

    return paths


# ── Illustrative simulation: 200 paths, 1 year ────────────────────────────────
S0, r, sigma, T = 100, 0.05, 0.25, 1.0
N_STEPS_PLOT = 252   # daily steps
paths_plot = simulate_gbm(S0, r, sigma, T, N_STEPS_PLOT, n_paths=200)

time_axis = np.linspace(0, T, N_STEPS_PLOT + 1)

fig, ax = plt.subplots(figsize=(13, 5))

# Plot individual paths with low alpha
ax.plot(time_axis, paths_plot[:, :50], lw=0.6, alpha=0.3, color="steelblue")

# Highlight 5th and 95th percentile bands
p5  = np.percentile(paths_plot, 5,  axis=1)
p95 = np.percentile(paths_plot, 95, axis=1)
ax.fill_between(time_axis, p5, p95, alpha=0.2, color="steelblue", label="5th–95th percentile")
ax.plot(time_axis, np.median(paths_plot, axis=1), color="navy", lw=2, label="Median path")
ax.axhline(S0, color="grey", lw=0.8, ls="--", label=f"S₀ = {S0}")

ax.set_title(f"GBM Simulated Stock Paths  |  S₀={S0}, r={r:.0%}, σ={sigma:.0%}, T={T}yr",
             fontweight="bold")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Stock Price (£)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nFinal price distribution (n=200 paths):")
final_prices = pd.Series(paths_plot[-1, :], name="Final Price")
print(final_prices.describe().to_string())


## 4. Monte Carlo Option Pricing — European Options

**Algorithm:**
1. Simulate $N$ terminal stock prices $S_T$ under the risk-neutral measure
2. Compute the payoff for each path: $\max(S_T - K, 0)$ for a call
3. Average the payoffs and discount back at the risk-free rate

$$C_{MC} = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} \max(S_T^{(i)} - K,\, 0)$$

This works for **any** payoff function — the real power of simulation.


In [ ]:

def mc_european(S: float, K: float, T: float, r: float,
                sigma: float, n_sims: int = 100_000,
                option: str = "call") -> dict:
    """
    Price a European option by Monte Carlo simulation (one-step, terminal price only).

    Returns dict with: price, std_error, ci_lower, ci_upper
    """
    # ── Simulate terminal stock prices (one-step trick — no need for full paths) ─
    Z  = np.random.standard_normal(n_sims)
    ST = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

    # ── Compute payoffs ───────────────────────────────────────────────────────
    if option == "call":
        payoffs = np.maximum(ST - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)

    # ── Discount and compute 95% confidence interval ──────────────────────────
    discount_factor = np.exp(-r * T)
    price      = discount_factor * payoffs.mean()
    std_error  = discount_factor * payoffs.std() / np.sqrt(n_sims)
    ci_lower   = price - 1.96 * std_error
    ci_upper   = price + 1.96 * std_error

    return {"price": price, "std_error": std_error,
            "ci_lower": ci_lower, "ci_upper": ci_upper}


# ── Compare MC vs BS for a range of strikes ───────────────────────────────────
S0, T, r, sigma = 100, 0.5, 0.05, 0.20
strikes = np.arange(70, 135, 5)

rows = []
for K in strikes:
    bs  = black_scholes(S=S0, K=K, T=T, r=r, sigma=sigma, option="call")
    mc  = mc_european(S=S0, K=K, T=T, r=r, sigma=sigma, n_sims=100_000, option="call")
    rows.append({
        "Strike": K,
        "BS Price":  bs["price"],
        "MC Price":  mc["price"],
        "MC Std Err": mc["std_error"],
        "Difference": mc["price"] - bs["price"],
        "Moneyness": "ITM" if S0 > K else ("ATM" if S0 == K else "OTM"),
    })

comparison = pd.DataFrame(rows).set_index("Strike")
print("Black-Scholes vs Monte Carlo (100,000 paths):")
comparison[["BS Price", "MC Price", "MC Std Err", "Difference", "Moneyness"]]


In [ ]:

# ── Visualise the convergence of MC price to BS price ─────────────────────────
# As we increase the number of simulation paths, MC converges to the analytical value

K_atm = 100   # At-the-money call
bs_price = black_scholes(S=S0, K=K_atm, T=T, r=r, sigma=sigma, option="call")["price"]

path_counts = [100, 500, 1_000, 2_000, 5_000, 10_000, 25_000, 50_000, 100_000]
n_repeats   = 30   # run 30 replicates at each path count to see variance

# ── Build results using list comprehension + method chaining ──────────────────
conv_results = (
    pd.DataFrame([
        {
            "n_paths": n,
            "replicate": rep,
            "mc_price": mc_european(S=S0, K=K_atm, T=T, r=r, sigma=sigma,
                                    n_sims=n, option="call")["price"]
        }
        for n in path_counts
        for rep in range(n_repeats)
    ])
    .assign(error=lambda df: df["mc_price"] - bs_price)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: price convergence ────────────────────────────────────────────────────
sns.lineplot(data=conv_results, x="n_paths", y="mc_price",
             estimator="mean", errorbar="sd",
             ax=axes[0], color="steelblue", lw=2)
axes[0].axhline(bs_price, color="red", lw=2, ls="--", label=f"BS price = {bs_price:.3f}")
axes[0].set_xscale("log")
axes[0].set_title("MC Price Convergence to BS Analytical Price", fontweight="bold")
axes[0].set_xlabel("Number of Simulated Paths (log scale)")
axes[0].set_ylabel("Option Price (£)")
axes[0].legend()

# ── Right: absolute error ──────────────────────────────────────────────────────
sns.lineplot(data=conv_results.assign(abs_error=lambda df: df["error"].abs()),
             x="n_paths", y="abs_error",
             estimator="mean", errorbar="sd",
             ax=axes[1], color="darkorange", lw=2)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Absolute Error vs Number of Paths", fontweight="bold")
axes[1].set_xlabel("Number of Simulated Paths (log scale)")
axes[1].set_ylabel("|MC Price − BS Price| (log scale)")

plt.suptitle("Monte Carlo Convergence Analysis  |  ATM Call, S=K=100, T=0.5yr, σ=20%",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: error shrinks as 1/√N — doubling accuracy requires 4× more paths.")


## 5. Path-Dependent Options — Asian Options

The real power of Monte Carlo is pricing options with **no closed-form solution**.

An **Asian call option** pays based on the *average* stock price over the life of the option:

$$\text{Payoff} = \max\left(\bar{S} - K,\, 0\right), \quad \bar{S} = \frac{1}{n}\sum_{t=1}^{n} S_t$$

This is widely used in commodity markets (e.g. oil derivatives) because a company buying
oil every month cares about the average price, not the price on one specific day.

There is no Black-Scholes equivalent formula — simulation is the primary pricing tool.


In [ ]:

def mc_asian_call(S: float, K: float, T: float, r: float,
                  sigma: float, n_steps: int = 252,
                  n_sims: int = 100_000) -> dict:
    """
    Price an arithmetic-average Asian call option by Monte Carlo.

    Parameters
    ----------
    n_steps : number of averaging points (e.g. 252 for daily averaging)

    Returns
    -------
    dict with price, std_error, ci_lower, ci_upper
    """
    # ── Simulate full price paths (shape: n_steps × n_sims) ──────────────────
    paths = simulate_gbm(S0=S, r=r, sigma=sigma, T=T,
                         n_steps=n_steps, n_paths=n_sims)

    # ── Average stock price for each path (column-wise mean, exclude t=0) ─────
    avg_price = paths[1:, :].mean(axis=0)   # arithmetic average of daily prices

    # ── Compute payoffs ───────────────────────────────────────────────────────
    payoffs = np.maximum(avg_price - K, 0)

    price     = np.exp(-r * T) * payoffs.mean()
    std_error = np.exp(-r * T) * payoffs.std() / np.sqrt(n_sims)

    return {
        "price":     price,
        "std_error": std_error,
        "ci_lower":  price - 1.96 * std_error,
        "ci_upper":  price + 1.96 * std_error,
    }


# ── Compare European vs Asian call prices across strikes ──────────────────────
S0, T, r, sigma = 100, 1.0, 0.05, 0.20

comparison_asian = (
    pd.DataFrame({"Strike": np.arange(80, 125, 5)})
    .assign(
        European_BS  = lambda df: df["Strike"].map(
            lambda K: black_scholes(S=S0, K=K, T=T, r=r, sigma=sigma, option="call")["price"]),
        Asian_MC     = lambda df: df["Strike"].map(
            lambda K: mc_asian_call(S=S0, K=K, T=T, r=r, sigma=sigma,
                                    n_steps=252, n_sims=50_000)["price"]),
    )
    .assign(Asian_Discount = lambda df: 1 - df["Asian_MC"] / df["European_BS"])
    .set_index("Strike")
)

print("European (BS) vs Asian (MC) Call Prices:")
print(comparison_asian.to_string())

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(comparison_asian.index, comparison_asian["European_BS"],
        "o-", lw=2, label="European Call (Black-Scholes)")
ax.plot(comparison_asian.index, comparison_asian["Asian_MC"],
        "s--", lw=2, label="Asian Call (Monte Carlo)")
ax.fill_between(comparison_asian.index,
                comparison_asian["Asian_MC"],
                comparison_asian["European_BS"],
                alpha=0.15, color="green", label="Asian discount")

ax.axvline(S0, color="grey", lw=1, ls=":", label="ATM (S=K=100)")
ax.set_title("European vs Asian Call Prices  |  S=100, σ=20%, T=1yr",
             fontweight="bold")
ax.set_xlabel("Strike Price (K)")
ax.set_ylabel("Option Price (£)")
ax.legend()
plt.tight_layout()
plt.show()

print("\nKey insight: Asian calls are CHEAPER than European calls.")
print("Averaging reduces volatility of the payoff — so you pay less for the option.")


## 6. Implied Volatility & The Volatility Smile

**Implied volatility (IV)** is the value of $\sigma$ that makes the Black-Scholes formula
reproduce a market-observed option price. It is found by **inverting** the BS formula numerically.

$$C_{\text{market}} = \text{BS}(S, K, T, r, \sigma_{\text{implied}})$$

If Black-Scholes were perfectly correct, IV would be constant across all strikes.
In reality, it forms a **volatility smile** or **skew** — evidence that BS assumptions
(constant vol, no jumps) are violated.


In [ ]:

def implied_vol(market_price: float, S: float, K: float, T: float,
                r: float, option: str = "call",
                tol: float = 1e-6) -> float:
    """
    Compute implied volatility by numerically inverting the BS formula.
    Uses Brent's method (scipy.optimize.brentq) — robust and fast.

    Returns float (implied vol) or np.nan if no solution found.
    """
    # ── Check if price is within no-arbitrage bounds ───────────────────────────
    intrinsic = max(S - K * np.exp(-r * T), 0) if option == "call" else max(K * np.exp(-r * T) - S, 0)
    if market_price < intrinsic:
        return np.nan

    # ── Objective function: BS price - market price = 0 ──────────────────────
    objective = lambda sigma: black_scholes(S, K, T, r, sigma, option)["price"] - market_price

    try:
        return brentq(objective, 1e-6, 10.0, xtol=tol)  # search vol in [0.0001, 1000%]
    except ValueError:
        return np.nan


# ── Simulate a volatility smile by constructing market prices with a skew ─────
# We 'generate' synthetic market prices using a skewed vol surface as ground truth,
# then recover the implied vol for each strike. This mimics real market behaviour.

S0, T, r = 100, 0.5, 0.05
strikes = np.arange(70, 135, 2.5)

def true_vol_smile(K, S0, atm_vol=0.20, skew=0.002):
    """ Simplified parametric smile: vol increases as we go OTM. """
    moneyness = np.log(K / S0)
    return atm_vol + skew * moneyness**2 - 0.001 * moneyness  # quadratic + skew

rows_smile = []
for K in strikes:
    true_sigma = true_vol_smile(K, S0)
    mkt_price  = black_scholes(S0, K, T, r, true_sigma, option="call")["price"]
    imp_vol    = implied_vol(mkt_price, S0, K, T, r, option="call")
    rows_smile.append({"Strike": K, "True Vol": true_sigma, "Implied Vol": imp_vol,
                       "Market Price": mkt_price})

smile_df = pd.DataFrame(rows_smile).set_index("Strike")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Volatility smile ─────────────────────────────────────────────────────
axes[0].plot(smile_df.index, smile_df["Implied Vol"] * 100, "o-", lw=2,
             color="steelblue", label="Implied Vol (recovered)")
axes[0].axhline(20, color="grey", lw=1, ls="--", label="Flat vol = 20%")
axes[0].axvline(S0, color="orange", lw=1, ls=":", label="ATM")
axes[0].set_title("The Volatility Smile", fontweight="bold")
axes[0].set_xlabel("Strike Price")
axes[0].set_ylabel("Implied Volatility (%)")
axes[0].legend()

# ── Right: Option prices showing skew ─────────────────────────────────────────
bs_flat_prices = [black_scholes(S0, K, T, r, 0.20, "call")["price"] for K in strikes]
axes[1].plot(strikes, smile_df["Market Price"], "o-", lw=2, label="Market prices (skewed vol)")
axes[1].plot(strikes, bs_flat_prices, "s--", lw=2, label="BS prices (flat 20% vol)")
axes[1].set_title("Skewed Market Prices vs Flat-Vol BS", fontweight="bold")
axes[1].set_xlabel("Strike Price")
axes[1].set_ylabel("Call Option Price (£)")
axes[1].legend()

plt.suptitle("Implied Volatility and the Volatility Smile", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Real Options — Valuing Managerial Flexibility

### The concept
Financial options give the **right, but not the obligation**, to buy/sell an asset.
**Real options** extend this logic to corporate investment decisions:

| Real Option | Financial Analogy | Example |
|-------------|------------------|---------|
| Option to **expand** | Call option | Invest more if project succeeds |
| Option to **abandon** | Put option | Sell/exit if market deteriorates |
| Option to **defer**   | Call option | Wait for more information before committing |
| Option to **switch**  | Exchange option | Switch fuel source if gas price rises |

### Limitations of applying BS to real options
Black-Scholes assumes:
- Continuous trading in the underlying asset ❌ (can't trade a factory)
- Constant volatility ❌ (project cash flow vol changes over time)
- No dividends / intermediate cash flows ❌ (projects produce cash flows)

Despite these limitations, BS gives useful **order-of-magnitude** estimates and the
intuition is invaluable for communicating option value to management.


In [ ]:

def real_option_expand(PV_asset: float, I_expand: float,
                       T_years: float, r_rf: float,
                       sigma_project: float) -> dict:
    """
    Value an 'option to expand' as a European call option using Black-Scholes.

    Parameters
    ----------
    PV_asset       : Present value of the project's future cash flows (= S)
    I_expand       : Additional investment required to expand (= K)
    T_years        : Window within which expansion decision must be made (= T)
    r_rf           : Risk-free rate
    sigma_project  : Volatility of the project's cash flows (estimate from comparables)

    Returns
    -------
    dict with: option_value, NPV_static, NPV_with_option
    """
    # ── Treat the option to expand as a call option ────────────────────────────
    call_result = black_scholes(
        S=PV_asset, K=I_expand, T=T_years, r=r_rf, sigma=sigma_project, option="call"
    )

    static_npv       = PV_asset - I_expand          # traditional static NPV
    npv_with_option  = static_npv + call_result["price"]  # expanded NPV

    return {
        "option_value":    call_result["price"],
        "NPV_static":      static_npv,
        "NPV_with_option": npv_with_option,
        "delta":           call_result["delta"],
    }


# ── Case Study: Pharmaceutical Company — Phase 2 Expansion Decision ───────────
print("=" * 60)
print("CASE STUDY: Pharma R&D — Option to Expand to Phase 2")
print("=" * 60)
print()
print("Background:")
print("  A pharma company completes Phase 1 of a drug trial.")
print("  If successful, they can invest £80m to expand to Phase 2.")
print("  The PV of Phase 2 cash flows (if launched) = £75m")
print("  They have 3 years to make the expansion decision.")
print("  Project cash flow volatility (from comparable drugs) ≈ 40%")
print()

result = real_option_expand(
    PV_asset=75,        # PV of Phase 2 cash flows
    I_expand=80,        # cost to expand
    T_years=3,          # decision window
    r_rf=0.04,          # risk-free rate
    sigma_project=0.40  # project volatility
)

print(f"  Static NPV (PV - Investment):       £{result['NPV_static']:.1f}m  ← REJECT by DCF")
print(f"  Option to Expand (BS value):        £{result['option_value']:.1f}m")
print(f"  Expanded NPV (static + option):     £{result['NPV_with_option']:.1f}m  ← ACCEPT with real options")
print()
print(f"  Option Delta: {result['delta']:.2f}  (hedge ratio — how sensitive value is to PV changes)")
print()
print("  Traditional DCF says REJECT. Real options analysis says the FLEXIBILITY")
print("  to expand has positive value — the project should not be abandoned yet.")


In [ ]:

# ── Sensitivity: how option value changes with volatility and time ─────────────
vol_range  = np.linspace(0.10, 0.80, 50)
time_range = [1, 2, 3, 5]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Option value vs project volatility ────────────────────────────────────────
for T_yr in time_range:
    vals = [real_option_expand(75, 80, T_yr, 0.04, sig)["option_value"]
            for sig in vol_range]
    axes[0].plot(vol_range * 100, vals, lw=2, label=f"T = {T_yr}yr")

axes[0].set_title("Option to Expand: Value vs Project Volatility", fontweight="bold")
axes[0].set_xlabel("Project Volatility (%)")
axes[0].set_ylabel("Real Option Value (£m)")
axes[0].legend(title="Decision window")

# ── Option value vs PV of asset (different project sizes) ────────────────────
pv_range = np.linspace(30, 140, 100)
for T_yr in [1, 3, 5]:
    vals = [real_option_expand(pv, 80, T_yr, 0.04, 0.40)["option_value"]
            for pv in pv_range]
    axes[1].plot(pv_range, vals, lw=2, label=f"T = {T_yr}yr")

axes[1].axvline(80, color="grey", lw=1, ls="--", label="I = £80m (ATM)")
axes[1].set_title("Option to Expand: Value vs PV of Asset", fontweight="bold")
axes[1].set_xlabel("PV of Future Cash Flows (£m)")
axes[1].set_ylabel("Real Option Value (£m)")
axes[1].legend(title="Decision window")

plt.suptitle("Real Option Sensitivity Analysis  |  Base: PV=75, K=80, σ=40%",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: HIGHER volatility → HIGHER option value.")
print("This is the opposite of DCF thinking — uncertainty creates option value!")


## 8. Bringing It Together — Real Vol from Market Data

Let's use the historical volatility from Session 4's data as the $\sigma$ input to Black-Scholes
and price options on AAPL at various strikes.


In [ ]:

# ── Download AAPL prices and compute realised vol ─────────────────────────────
aapl = (
    yf.download("AAPL", start="2023-01-01", end="2024-12-31",
                auto_adjust=True, progress=False)["Close"]
    .squeeze()
    .rename("AAPL")
    .pipe(lambda s: np.log(s / s.shift(1)))  # log returns
    .dropna()
)

# ── Use last 30-day realised vol as our sigma estimate ────────────────────────
TRADING_DAYS = 252
sigma_30d = aapl.tail(30).std() * np.sqrt(TRADING_DAYS)
S_current = yf.download("AAPL", period="1d", auto_adjust=True, progress=False)["Close"].iloc[-1]
r_current = 0.05   # approximate 10-year UST yield

print(f"AAPL Last Price    : ${S_current:.2f}")
print(f"30-day Realised Vol: {sigma_30d:.1%}")
print(f"Risk-free Rate     : {r_current:.1%}")
print()

# ── Price options at ±20% strikes for 1-month, 3-month, 6-month expiry ────────
expiries = {"1 Month": 1/12, "3 Months": 3/12, "6 Months": 6/12}
K_multipliers = np.array([0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20])

rows_live = []
for label, T_yr in expiries.items():
    for km in K_multipliers:
        K = round(float(S_current) * km, 1)
        bs_c = black_scholes(float(S_current), K, T_yr, r_current, sigma_30d, "call")
        bs_p = black_scholes(float(S_current), K, T_yr, r_current, sigma_30d, "put")
        rows_live.append({
            "Expiry": label, "Strike": K, "K/S": f"{km:.0%}",
            "Call": bs_c["price"], "Put": bs_p["price"],
            "Call Delta": bs_c["delta"], "Call Vega": bs_c["vega"],
        })

live_options = pd.DataFrame(rows_live)

# ── Pivot to option chain style display ───────────────────────────────────────
option_chain = (
    live_options
    .query("Expiry == '3 Months'")
    .assign(Moneyness=lambda df: df["K/S"].map({
        "80%": "Deep ITM", "85%": "ITM", "90%": "ITM",
        "95%": "Slight ITM", "100%": "ATM",
        "105%": "Slight OTM", "110%": "OTM", "115%": "OTM", "120%": "Deep OTM"
    }))
    [["Strike", "K/S", "Call", "Put", "Call Delta", "Call Vega", "Moneyness"]]
    .set_index("Strike")
    .round(3)
)

print("AAPL Option Chain — 3-Month Expiry (BS Theoretical Prices)")
print(option_chain.to_string())


## Summary & Key Takeaways

| Topic | Key Insight |
|-------|-------------|
| Black-Scholes | Closed-form, fast, elegant — but rests on strong assumptions |
| Greeks | Delta = hedge ratio; Gamma = acceleration; Vega = vol sensitivity; Theta = time decay |
| Monte Carlo | Flexible — can price ANY payoff; converges at rate 1/√N |
| Asian options | Path-dependent; no closed form; always cheaper than European |
| Implied vol smile | Markets reject flat-vol BS; smile = fat tails + skew built into prices |
| Real options | Corporate flexibility (expand/abandon/defer) has quantifiable option value |
| Vol estimation | 30-day realised vol is a simple but imperfect estimate of future vol |

---

### Limitations of Black-Scholes

1. **Constant volatility** — volatility actually clusters and smiles (Session 4 showed this)
2. **Continuous trading** — impossible for real assets, illiquid instruments
3. **Log-normal distribution** — ignores fat tails and jump risk (crashes)
4. **No transaction costs** — delta hedging in practice incurs costs
5. **European exercise only** — American options require binomial trees or PDE methods

These limitations motivate more advanced models: **Heston** (stochastic vol),
**SABR** (smile modelling), and **Jump-Diffusion** (Merton model) — natural next steps for further study.
